In [3]:
import numpy as np 
import pandas as pd 
import statistics as sts 
from matplotlib import pyplot as plt 
import seaborn as sns
from sklearn.preprocessing import StandardScaler,LabelEncoder,OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from scipy.spatial import distance as dts
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error,root_mean_squared_error,median_absolute_error
from sklearn.pipeline import Pipeline


In [4]:
def Sample_calculations(df2):
    print(df2.ndim)
    print(df2.size)
    print(df2.shape)
    print(df2.describe())
    print(df2.info())
    print(df2.columns)    



try:
    df1=pd.read_csv(r"F:\ExternalSoftwares\pythoncodes\MachineLearning\Salary Data.csv",encoding_errors='ignore')
    df2=df1.copy()
    Sample_calculations(df2)
except Exception as e:
    print(f"{str(e)}")
else:
    print('no errors')
finally:
    print("final block1")

2
2250
(375, 6)
              Age  Years of Experience         Salary
count  373.000000           373.000000     373.000000
mean    37.431635            10.030831  100577.345845
std      7.069073             6.557007   48240.013482
min     23.000000             0.000000     350.000000
25%     31.000000             4.000000   55000.000000
50%     36.000000             9.000000   95000.000000
75%     44.000000            15.000000  140000.000000
max     53.000000            25.000000  250000.000000
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 375 entries, 0 to 374
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Age                  373 non-null    float64
 1   Gender               373 non-null    object 
 2   Education Level      373 non-null    object 
 3   Job Title            373 non-null    object 
 4   Years of Experience  373 non-null    float64
 5   Salary               373 non-null    fl

In [5]:
def preprossing_data(df2):
    print(df2['Gender'].value_counts().to_dict())
    print(df2['Education Level'].value_counts().to_dict())

preprossing_data(df2)

{'Male': 194, 'Female': 179}
{"Bachelor's": 224, "Master's": 98, 'PhD': 51}


In [6]:
print(df2.isna().sum())

Age                    2
Gender                 2
Education Level        2
Job Title              2
Years of Experience    2
Salary                 2
dtype: int64


In [7]:
df2.dropna(axis=0,how='any',inplace=True)

In [8]:
df2.drop(columns=['Job Title'],axis=1,inplace=True)

In [9]:
df2.drop_duplicates(inplace=True)

In [10]:
df2=df2.astype({'Age':'int','Years of Experience':'int','Salary':'int'})

{'Male': 194, 'Female': 179}
{"Bachelor's": 224, "Master's": 98, 'PhD': 51}

In [11]:
gender_mapping={
    'Male':1,
    'Female':2
}
Education_level_mapping={
    "Bachelor's":1,
    "Master's":2,
    "PhD":3
}
df2['gender_encoded']=df2['Gender'].map(gender_mapping)
df2['Education_Level_encoded']=df2['Education Level'].map(Education_level_mapping)

In [12]:
x=df2[['Age','gender_encoded','Education_Level_encoded']]
y=df2['Salary']
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

linear_regression=LinearRegression()
linear_regression.fit(x_train,y_train)
y_pridaction=linear_regression.predict(x_test)
print(y_pridaction)

import joblib
joblib.dump(linear_regression,'salary_predication.joblib')

[158913.23400154 112002.96613558  83313.94919789 116062.9278884
  73234.04455741  93393.85383838  68194.09223716  97453.8155912
  92413.86327096  87373.91095072 121102.88020865  47054.29238877
  42014.34006853  77294.00631023  96883.10917485 149791.89714633
  73234.04455741  97453.8155912   97453.8155912   67214.10166974
 141262.68948962  93393.85383838  57134.19702926 170931.69699473
 167442.44165825  83313.94919789  62174.1493495  132162.77541655
 135652.03075302 192642.20325947  58114.18759668 154831.84946658
 132162.77541655  37954.37831571  42994.33063595  86803.20453436
  92413.86327096  57134.19702926 126142.83252889  73234.04455741
 135652.03075302  53074.23527643  96883.10917485  36974.38774829
 101923.06149509 112002.96613558 163953.18632178 154831.84946658
 143793.37704081 159871.80178682  52094.24470901  97453.8155912
  53074.23527643 128673.52008008  58114.18759668]


['salary_predication.joblib']

In [13]:
x

,Age,gender_encoded,Education_Level_encoded
0,32,1,1
1,28,2,2
2,45,1,3
3,36,2,1
4,52,1,2
...,...,...,...
344,44,2,3
347,45,1,2
348,28,2,1
349,36,1,1


In [ ]:
import streamlit as st 
import pandas as pd 
import joblib


try:
    model=joblib.load(r'salary_predication.joblib')
    st.title("Employee Salary predications")
    st.write("Model is loaded succesfully ....!")
    st.balloons()

    st.sidebar.write("Enter the input parameters")
    with st.form(key="input-values"):
        with st.sidebar:
            Age=st.number_input('age',min_value=19,max_value=65,value=35,help='age is between 19 to 65')
            Gender=st.number_input('gender',min_value=1,max_value=2,value=1,help='male is 1 and female is 2')
            Education_level=st.number_input('Education',min_value=1,max_value=3,value=2,help='1 -> is Bachlors  2 -> is for masters and 3 --> is for PHD')
            submit_button=st.form_submit_button(label='Submit')

    if submit_button:
        data1={
            'Age': Age,
            'gender_encoded': Gender,
            'Education_Level_encoded':Education_level
        }            

        df=pd.DataFrame(data=data1,index=[0])
        predications=model.predict(df)
        print(f"the predication salary is {predications:,.2f} ")
        print("Thank You .....!")
        st.balloons()


except Exception as e:
    st.error(f"the error is {e}")
    st.code(f"{str(e)} ", language='python')

else:
    st.write("no errors in the model")

finally:
    st.write(" Finaly Block is exceuted succsufully")
    
    

_IncompleteInputError: incomplete input (2341928520.py, line 7)

In [ ]:
# input_numerical = ['Age', 'Years of Experience']
# input_categorical = ['Gender', 'Education Level']  # Fixed typo: "catagerical" → "categorical"

# numerical_features = Pipeline(steps=[
#     ('imputer', SimpleImputer(strategy='median')),
#     ('scaler', StandardScaler())
# ])

# categorical_features = Pipeline(steps=[
#     ('imputer', SimpleImputer(strategy='constant', fill_value='missing_data')),
#     ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))  # ✅ Fixed
# ])

# columns_transformer = ColumnTransformer(
#     transformers=[
#         ('num', numerical_features, input_numerical),
#         ('cat', categorical_features, input_categorical)
#     ]
# )

# # Fit and transform
# df2_processed = columns_transformer.fit_transform(df2) 
# df2_processed